# 03 — Agent 1 : sélection de l'échelle et détection des caractéristiques

L'Agent 1 ne doit pas "inventer" une pente ou un lacet.

Son rôle est :

```text
caractéristique cible
    ↓
choix du détecteur
    ↓
choix d'une ou plusieurs fenêtres spatiales
    ↓
exécution Python déterministe
    ↓
annotations atomiques
```

Dans cette première version, on implémente réellement :
- pente ;
- lacets ;
- sinuosité.

Les autres caractéristiques seront ajoutées après validation du pipeline.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import requests

## 0. Charger la route normalisée produite par le notebook 02

Le notebook 03 est autonome : il recharge la route depuis le disque.

Avant de continuer, le notebook 02 doit avoir créé :

```text
data/output/route_resampled.csv
```

In [ ]:
route_path = Path("data/output/route_resampled.csv")

if not route_path.exists():
    raise FileNotFoundError(
        "Route normalisée introuvable. Exécute d'abord le notebook 02 "
        "jusqu'à la sauvegarde de data/output/route_resampled.csv."
    )

route = pd.read_csv(route_path)

required_columns = {"distance_m", "lat", "lon", "elev_m"}
missing = required_columns - set(route.columns)

if missing:
    raise ValueError(f"Colonnes manquantes dans la route : {sorted(missing)}")

print("Route chargée :", route_path)
print("Nombre de points :", len(route))
print("Longueur :", round(route["distance_m"].iloc[-1], 1), "m")
display(route.head())

## 1. Configuration des caractéristiques

L'Agent 1 n'a pas le droit de proposer n'importe quelle fenêtre.
Il choisit dans une liste autorisée.

In [ ]:
FEATURE_CONFIG = {
    "forte_pente": {
        "detector": "slope",
        "allowed_windows_m": [10, 25, 50],
    },
    "pente_douce": {
        "detector": "slope",
        "allowed_windows_m": [25, 50, 100],
    },
    "montee": {
        "detector": "slope",
        "allowed_windows_m": [50, 100, 200],
    },
    "descente": {
        "detector": "slope",
        "allowed_windows_m": [50, 100, 200],
    },
    "lacets": {
        "detector": "switchbacks",
        "allowed_windows_m": [75, 150, 250],
    },
    "sinuosite": {
        "detector": "sinuosity",
        "allowed_windows_m": [50, 100, 200],
    },
}

## 2. Détecteur de pente

Pente en pourcentage :

```text
pente_pct = 100 × Δaltitude / Δdistance
```

Le calcul est déterministe.

In [ ]:
def compute_window_slopes(route, window_m=25, step_m=10):
    distances = route["distance_m"].to_numpy()
    elev = route["elev_m"].to_numpy()

    rows = []
    start = 0.0
    end_total = distances[-1]

    while start + window_m <= end_total:
        end = start + window_m

        z0 = np.interp(start, distances, elev)
        z1 = np.interp(end, distances, elev)

        slope_pct = 100.0 * (z1 - z0) / window_m

        rows.append({
            "start_m": float(start),
            "end_m": float(end),
            "slope_pct": float(slope_pct),
        })

        start += step_m

    return pd.DataFrame(rows)

In [ ]:
def detect_slope_feature(route, feature, window_m):
    slopes = compute_window_slopes(route, window_m=window_m, step_m=10)

    if feature == "forte_pente":
        mask = slopes["slope_pct"] >= 12
    elif feature == "pente_douce":
        mask = (slopes["slope_pct"] >= 3) & (slopes["slope_pct"] < 8)
    elif feature == "montee":
        mask = slopes["slope_pct"] >= 5
    elif feature == "descente":
        mask = slopes["slope_pct"] <= -5
    else:
        raise ValueError(feature)

    out = slopes.loc[mask].copy()
    out["feature"] = feature
    out["resolution_m"] = window_m
    out["confidence"] = np.minimum(1.0, np.abs(out["slope_pct"]) / 20.0)
    return out

## 3. Détecteur de lacets

Prototype V1 :

- calcul de l'angle de direction entre points successifs ;
- comptage des changements de direction importants dans une fenêtre ;
- détection des alternances gauche/droite.

Ce détecteur devra être raffiné scientifiquement plus tard.

In [ ]:
def bearing_deg(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dlambda = math.radians(lon2 - lon1)

    y = math.sin(dlambda) * math.cos(phi2)
    x = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(dlambda)
    )
    return (math.degrees(math.atan2(y, x)) + 360) % 360


def angular_difference(a, b):
    return (b - a + 180) % 360 - 180


def detect_switchbacks(route, window_m=150, step_m=25, min_turn_deg=35, min_turns=3):
    d = route["distance_m"].to_numpy()
    lat = route["lat"].to_numpy()
    lon = route["lon"].to_numpy()

    bearings = np.array([
        bearing_deg(lat[i], lon[i], lat[i+1], lon[i+1])
        for i in range(len(route)-1)
    ])

    turns = np.array([
        angular_difference(bearings[i], bearings[i+1])
        for i in range(len(bearings)-1)
    ])

    turn_distance = d[1:-1]
    results = []

    start = 0.0
    while start + window_m <= d[-1]:
        end = start + window_m
        idx = (turn_distance >= start) & (turn_distance <= end)

        local_turns = turns[idx]
        significant = local_turns[np.abs(local_turns) >= min_turn_deg]

        signs = np.sign(significant)
        alternations = (
            np.sum(signs[1:] * signs[:-1] < 0)
            if len(signs) >= 2 else 0
        )

        if len(significant) >= min_turns and alternations >= 2:
            score = min(1.0, 0.2 * len(significant) + 0.15 * alternations)
            results.append({
                "feature": "lacets",
                "start_m": float(start),
                "end_m": float(end),
                "resolution_m": float(window_m),
                "confidence": float(score),
                "direction_changes": int(len(significant)),
                "alternations": int(alternations),
            })

        start += step_m

    return pd.DataFrame(results)

## 4. Détecteur de sinuosité

On compare :

```text
longueur parcourue / distance directe
```

Une valeur proche de 1 indique un tracé assez direct.
Une valeur plus élevée indique un tracé sinueux.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))


def detect_sinuosity(route, window_m=100, step_m=25, threshold=1.15):
    d = route["distance_m"].to_numpy()
    results = []

    start = 0.0
    while start + window_m <= d[-1]:
        end = start + window_m

        i0 = int(np.argmin(np.abs(d - start)))
        i1 = int(np.argmin(np.abs(d - end)))

        direct = haversine_m(
            route.loc[i0, "lat"], route.loc[i0, "lon"],
            route.loc[i1, "lat"], route.loc[i1, "lon"]
        )

        sinuosity = window_m / max(direct, 1e-6)

        if sinuosity >= threshold:
            results.append({
                "feature": "sinuosite",
                "start_m": float(start),
                "end_m": float(end),
                "resolution_m": float(window_m),
                "confidence": float(min(1.0, (sinuosity - 1.0) / 0.5)),
                "sinuosity": float(sinuosity),
            })

        start += step_m

    return pd.DataFrame(results)

## 5. Client Ollama minimal

Le LLM ne calcule rien.
Il choisit seulement les fenêtres à tester parmi celles autorisées.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b"

def ask_ollama_json(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "format": "json",
            "options": {"temperature": 0},
        },
        timeout=120,
    )
    response.raise_for_status()
    content = response.json()["message"]["content"]
    return json.loads(content)

## 6. Agent 1 : choix des fenêtres

Si Ollama n'est pas disponible, le notebook peut fonctionner en mode déterministe de secours :
il teste toutes les fenêtres autorisées.

In [ ]:
def choose_windows(feature, route, use_llm=True):
        cfg = FEATURE_CONFIG[feature]
        allowed = cfg["allowed_windows_m"]

        if not use_llm:
            return allowed

        prompt = f'''
Tu es un agent d'orchestration géospatiale.

Caractéristique à détecter : {feature}
Détecteur disponible : {cfg["detector"]}
Fenêtres autorisées en mètres : {allowed}
Longueur de l'itinéraire : {route["distance_m"].iloc[-1]:.1f} m
Résolution de base : 10 m

Tu ne calcules aucune métrique géométrique.
Tu choisis seulement une ou plusieurs fenêtres parmi la liste autorisée.

Réponds uniquement avec un JSON de cette forme :
{{"windows_to_test_m": [150]}}
'''

        try:
            data = ask_ollama_json(prompt)
            chosen = [
                int(w) for w in data["windows_to_test_m"]
                if int(w) in allowed
            ]
            return chosen or allowed
        except Exception as exc:
            print("Ollama indisponible ou réponse invalide :", exc)
            print("Fallback : test de toutes les fenêtres autorisées.")
            return allowed

## 7. Exécuter un détecteur

Cette fonction relie :

```text
caractéristique
→ Agent 1
→ fenêtres
→ détecteur Python
```

In [ ]:
def run_feature_detection(route, feature, use_llm=True):
    windows = choose_windows(feature, route, use_llm=use_llm)
    detector = FEATURE_CONFIG[feature]["detector"]

    frames = []

    for window in windows:
        if detector == "slope":
            df = detect_slope_feature(route, feature, window)
        elif detector == "switchbacks":
            df = detect_switchbacks(route, window_m=window)
        elif detector == "sinuosity":
            df = detect_sinuosity(route, window_m=window)
        else:
            raise ValueError(detector)

        if not df.empty:
            frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

## 8. Test réel de l'Agent 1

On teste maintenant toutes les caractéristiques définies dans `FEATURE_CONFIG`.

Deux modes sont possibles :

```python
USE_LLM = False
```

→ pas de LLM : toutes les fenêtres autorisées sont testées.

```python
USE_LLM = True
```

→ Ollama/Llama choisit une ou plusieurs fenêtres avant l'exécution des détecteurs.

Pour le premier passage, commencez par `False` afin de valider la partie géométrique.
Passez ensuite à `True`.

In [ ]:
USE_LLM = False

FEATURES_TO_TEST = [
    "forte_pente",
    "pente_douce",
    "montee",
    "descente",
    "lacets",
    "sinuosite",
]

detection_results = {}

for feature in FEATURES_TO_TEST:
    print("\n" + "=" * 70)
    print("CARACTÉRISTIQUE :", feature)

    windows = choose_windows(
        feature,
        route,
        use_llm=USE_LLM
    )

    print("Fenêtre(s) choisie(s) :", windows)

    # On exécute explicitement les détecteurs avec les fenêtres choisies
    detector = FEATURE_CONFIG[feature]["detector"]
    frames = []

    for window in windows:
        if detector == "slope":
            df = detect_slope_feature(route, feature, window)

        elif detector == "switchbacks":
            df = detect_switchbacks(route, window_m=window)

        elif detector == "sinuosity":
            df = detect_sinuosity(route, window_m=window)

        else:
            raise ValueError(f"Détecteur inconnu : {detector}")

        if not df.empty:
            frames.append(df)

    if frames:
        result = pd.concat(frames, ignore_index=True)
    else:
        result = pd.DataFrame()

    detection_results[feature] = result

    print("Nombre de détections :", len(result))

    if not result.empty:
        display(result.head(10))
    else:
        print("Aucune portion détectée pour cette caractéristique.")

## 9. Résumé des résultats

Ce tableau donne le nombre d'annotations atomiques produites par caractéristique.

In [ ]:
summary = pd.DataFrame([
    {
        "feature": feature,
        "n_detections": len(df),
        "resolutions_m": (
            sorted(df["resolution_m"].unique().tolist())
            if not df.empty and "resolution_m" in df.columns
            else []
        ),
    }
    for feature, df in detection_results.items()
])

display(summary)

## 10. Activer maintenant le LLM

Une fois le test déterministe validé, remplacez :

```python
USE_LLM = False
```

par :

```python
USE_LLM = True
```

Relancez uniquement le bloc de test.

L'Agent 1 demandera alors à Ollama de choisir la ou les fenêtres parmi les valeurs autorisées.

### Important

Cette V1 de l'Agent 1 choisit l'échelle **avant** de voir les résultats des détecteurs.

La version de recherche suivante devra être plus intéressante :

```text
Agent 1
   ↓
propose plusieurs échelles
   ↓
Python exécute les détecteurs
   ↓
métriques comparées
   ↓
Agent 1 sélectionne/interprète la meilleure échelle
```

Ce sera la prochaine évolution après validation du pipeline actuel.